# 02. Deque and Monotonic Deque

A **deque** (double-ended queue) adds and removes at **both ends** in O(1). A **monotonic deque** keeps values in order so the **maximum or minimum of a sliding window** is always at the front.

**Learning order:** DEQUE → IDEA → SLIDING WINDOW MAXIMUM → FIRST NEGATIVE → LONGEST SUBARRAY WITH LIMIT → SHORTEST SUBARRAY SUM AT LEAST K → INTERVIEW THINKING

## 1. Deque in Python

`append`, `appendleft`, `pop`, `popleft`, all O(1). Also `rotate(n)`. Indexing in the middle is O(n), so use it at the ends.

In [1]:
from collections import deque

d = deque([2, 3])
d.appendleft(1); d.append(4)
print(list(d), d.pop(), d.popleft())   # [1, 2, 3, 4] 4 1

[1, 2, 3, 4] 4 1


## 2. Sliding Window Maximum

**Problem:** maximum of every window of size `k`. Brute force is O(n·k).

**Idea:** keep a deque of **indexes** whose values are **decreasing**.
- When a new `x` arrives, pop from the **back** every index with value `<= x`. They can never be a maximum while `x` is in the window (x is bigger and stays longer).
- Pop from the **front** if that index has left the window (`index <= i - k`).
- The **front** is the maximum of the current window.

Real life: "highest temperature in the last 7 days", updated every day.

Each index is pushed and popped once → **O(n)**.

In [2]:
def sliding_window_max(nums, k):
    dq, res = deque(), []                        # dq holds indexes, values decreasing
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:                       # front is out of the window
            dq.popleft()
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

print(sliding_window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))   # [3, 3, 5, 5, 6, 7]

[3, 3, 5, 5, 6, 7]


## 3. First negative number in every window

Keep a deque of indexes of **negative numbers** only. Remove the front if it is outside the window. The front (if any) is the first negative; otherwise answer `0`.

In [3]:
def first_negative_in_windows(nums, k):
    dq, res = deque(), []
    for i, x in enumerate(nums):
        if x < 0:
            dq.append(i)
        if i >= k - 1:
            while dq and dq[0] <= i - k:
                dq.popleft()
            res.append(nums[dq[0]] if dq else 0)
    return res

print(first_negative_in_windows([12, -1, -7, 8, -15, 30, 16, 28], 3))   # [-1, -1, -7, -15, -15, 0]

[-1, -1, -7, -15, -15, 0]


## 3b. Longest subarray with `max - min <= limit`

Use **two deques** over a sliding window: one decreasing (for the max), one increasing (for the min). If `max - min > limit`, move `left` forward (and drop deque fronts that left the window).

In [4]:
def longest_subarray(nums, limit):
    maxq, minq = deque(), deque()                # indexes
    left = best = 0
    for right, x in enumerate(nums):
        while maxq and nums[maxq[-1]] <= x:
            maxq.pop()
        maxq.append(right)
        while minq and nums[minq[-1]] >= x:
            minq.pop()
        minq.append(right)
        while nums[maxq[0]] - nums[minq[0]] > limit:
            left += 1
            if maxq[0] < left: maxq.popleft()
            if minq[0] < left: minq.popleft()
        best = max(best, right - left + 1)
    return best

print(longest_subarray([8, 2, 4, 7], 4), longest_subarray([10, 1, 2, 4, 7, 2], 5), longest_subarray([4, 2, 2, 2, 4, 4, 2, 2], 0))   # 2 4 3

2 4 3


## 4. Shortest Subarray with Sum at least K (with negatives)

Sliding window fails with negatives (see Sliding Window notebook). Use **prefix sums** and a deque of indexes with **increasing** prefix values.

- While `prefix[i] - prefix[front] >= k`: we found a valid subarray, record `i - front`, pop the front (a later `i` would only give a longer subarray).
- Before pushing `i`: pop from the back while `prefix[back] >= prefix[i]` (a smaller, later prefix is always better as a start).

In [5]:
def shortest_subarray(nums, k):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    dq, best = deque(), float("inf")
    for i, p in enumerate(prefix):
        while dq and p - prefix[dq[0]] >= k:
            best = min(best, i - dq.popleft())
        while dq and prefix[dq[-1]] >= p:
            dq.pop()
        dq.append(i)
    return -1 if best == float("inf") else best

print(shortest_subarray([1], 1), shortest_subarray([1, 2], 4), shortest_subarray([2, -1, 2], 3))   # 1 -1 3

1 -1 3


## 5. Interview thinking

**Signals:** "maximum/minimum of every window", "window with a max-min condition", "shortest subarray with sum ≥ k and negatives".

**Say:** "I keep a deque of indexes in monotonic order. The front is the answer. I drop expired indexes from the front and dominated indexes from the back. Each index enters and leaves once: O(n)."

**Common mistakes:** storing values instead of indexes (can't expire them), wrong expiry check (`<= i - k`), popping with `<` vs `<=`, forgetting to start recording only after the first full window.

**Alternative:** a heap works in O(n log n) with lazy deletion. The deque is faster and the expected answer.

## Quick Revision

- Deque: O(1) at both ends.
- Window max: decreasing deque of indexes, front = max, expire front by index.
- Two deques for max-min windows.
- Negatives + shortest sum ≥ k: prefix sums + increasing deque.
- O(n) because each index is pushed/popped once.